# 02 — Inference + submit (Kaggle, self-contained)

Run this **second** (after `01_train_lora.ipynb`), or run it alone with `HF_ADAPTER_REPO = None` for base-model predictions.

Clones the repo from GitHub, loads Qwen2.5-VL-7B (4-bit Unsloth) + optional LoRA adapter from HuggingFace Hub, runs sync detect → transcribe on the test set, and writes `/kaggle/working/submission.csv`.

The competition ships only `sample_submission.csv` (no images) — the test pages are pulled from the HF dataset's `test` split at inference time, the same way notebook 01 pulls train/silver.

**Setup:**
- Right sidebar → _+ Add Data_ → attach `handwritten-to-data` (gives `sample_submission.csv`).
- Internet: On.
- `HF_TOKEN` secret (Kaggle → Add-ons → Secrets, toggle Attached) — only if your adapter repo is private. The `rukopys` dataset itself is public.

**Submitting:** this is a **code competition** — submission is by committing the notebook, not the Kaggle API. Save Version → **Save & Run All (Commit)**; when the version finishes, open it → Output tab → **Submit to Competition**. The committed run's `submission.csv` is what gets scored.

In [ ]:
# 1. Install GPU stack.
%pip -q install --upgrade unsloth unsloth_zoo
%pip -q install --no-deps trl peft accelerate bitsandbytes
%pip -q install pyyaml opencv-python-headless

In [ ]:
GITHUB_REPO     = "https://github.com/dhodyrev/handwritten-to-data.git"
HF_ADAPTER_REPO = "dkhodyriev/htr-qwen25vl-transcribe-lora"  # set to None for base-model inference
CONFIG          = "configs/pipeline.yaml"                     # or configs/pipeline_p1.yaml

In [ ]:
# 3. Clone repo into /kaggle/working/repo and install htr.
import os, subprocess, sys
REPO_DIR = "/kaggle/working/repo"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", GITHUB_REPO, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/src")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"], check=True)
print("cwd:", os.getcwd())

In [ ]:
# 4. HuggingFace login — only needed if the adapter repo is private.
if HF_ADAPTER_REPO:
    from huggingface_hub import login
    try:
        from kaggle_secrets import UserSecretsClient
        login(token=UserSecretsClient().get_secret("HF_TOKEN"))
        print("hf login OK")
    except Exception as e:
        print(f"hf login skipped ({e}); proceeding anonymously (only works if adapter is public)")

In [ ]:
# 5. Build the test manifest. The competition ships only sample_submission.csv
#    (no images, bare '<uuid>.jpg' ids). The pages live in the HF dataset's
#    'test' split at 'test/images/<uuid>.jpg' — its metadata.jsonl carries both
#    the 'images/'-prefixed file_name AND the real source (school/university/
#    dictation/archive), so we join on it to get correct paths + source-aware
#    routing instead of a blanket 'unknown'.
import csv, json, glob, pathlib
from htr.data import iter_metadata

hits = glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)
if not hits:
    avail = sorted(glob.glob("/kaggle/input/**/*", recursive=True))[:40] or ["(nothing mounted)"]
    raise FileNotFoundError(f"sample_submission.csv not found. Mounted (first 40): {avail}")
SAMPLE = hits[0]
with open(SAMPLE) as f:
    sub_images = [row["image"] for row in csv.DictReader(f)]

# HF test metadata keyed by bare basename → {file_name (images/<uuid>.jpg), source}
meta = {pathlib.Path(m["file_name"]).name: m for m in iter_metadata("test")}

manifest = pathlib.Path("data/cv/test.jsonl")
manifest.parent.mkdir(parents=True, exist_ok=True)
missing = 0
with manifest.open("w") as out:
    for img in sub_images:
        m = meta.get(pathlib.Path(img).name)
        if m:
            fn, src = m["file_name"], m.get("source", "unknown")
        else:  # fallback: assume the standard images/ layout
            fn, src = f"images/{img}", "unknown"
            missing += 1
        out.write(json.dumps({"file_name": fn, "source": src}) + "\n")
print(f"wrote {len(sub_images)} test items to {manifest}"
      + (f"  ({missing} not in HF metadata — used images/ fallback)" if missing else ""))

In [ ]:
# 6. Run inference. Images are auto-downloaded from the HF 'test' split
#    (--hf-split test) and cached, exactly like notebook 01 pulls train/silver.
#    backend.load_model passes the adapter string through to from_pretrained,
#    which resolves HF Hub IDs the same as local paths.
adapter_arg = f"--adapter {HF_ADAPTER_REPO}" if HF_ADAPTER_REPO else ""
!python scripts/run_inference.py \
    --manifest data/cv/test.jsonl \
    --hf-split test \
    --config {CONFIG} \
    {adapter_arg} \
    --out /kaggle/working/submission.csv

In [ ]:
# 7. Spot-check the submission.
import csv
with open("/kaggle/working/submission.csv") as f:
    rows = list(csv.DictReader(f))
print(f"rows: {len(rows)}")
print("first image:", rows[0]["image"])
print("regions head:", rows[0]["regions"][:300], "...")

In [ ]:
# 8. Submit. This is a CODE competition (data mounts under
#    /kaggle/input/competitions/...), so submission is by *committing the
#    notebook*, not the Kaggle API — the API rejects submissions here, and the
#    kernel has no credentials anyway. A committed run's /kaggle/working/
#    submission.csv is what gets scored.
#
#    To submit:  Save Version → "Save & Run All (Commit)". When it finishes,
#    open the version → Output tab → "Submit to Competition".
import pathlib, csv

SUB = pathlib.Path("/kaggle/working/submission.csv")
assert SUB.exists(), f"{SUB} missing — run the inference cell (6) first."
with SUB.open() as f:
    rows = list(csv.DictReader(f))
assert rows, "submission.csv is empty."
print(f"submission.csv ready: {len(rows)} rows, columns={list(rows[0])}")
print("→ Save Version (Commit), then submit from the version's Output tab.")